# 1. Package Imports

In [0]:
import re
import logging
import pyspark.sql.functions as F
from pyspark.sql import Window

# 2. Dataset Configs

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_Dam_Levels")
# logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_CPI")

#silver tables config
ds_config = {
        "silver_table": "cpt_utility_catalog.silver.silver_subcouncils_arrears_cleaned",
        "bronze_table": "cpt_utility_catalog.bronze.bronze_subcouncils_arrears_raw",
        "changes": {
            "headers": {
                "column_mapping":{
                    "Subcouncil": "subcouncil",
                    "B__30___1_": "credit",
                    "C_0_29": "0_29",
                    "D_30_59": "30_59",
                    "E_60_89": "60_89",
                    "F_90_119": "90_119",
                    "G_120_149": "120_149",
                    "H_150_179": "150_179",
                    "I_180_364": "180_364",
                    "J_365_729": "365_729",
                    "K_730_1094": "730_1094",
                    "L_1095_1459": "1095_1459",
                    "M_1460_1824": "1460_1824",
                    "N_1825_2189": "1825_2189",
                    "O_2190_2554": "2190_2554",
                    "P_2555_2919": "2555_2919",
                    "Q_2920_3284": "2920_3284",
                    "R___3285": "3285_plus"
                }
               
            },
            "columns": {
                 "data_types":{
                    "amount": "decimal(13,2)",
                    "date": "date",
                    "ageing_bucket_days": "string",
                    "id": "string",
                    "subcouncil": "int",
                },
                "columns_to_drop": ["ObjectId","Result"],
                "fill_na_value": None, 
            },
            "trim": True,
            "drop_columns": True,
            "drop_duplicates": True,
            "write_to_table": True,
            "rename_headers": True,
            "unpivot": True,
            "add_id": True,
            "cast_type": True,
            "col_cleanse": True,
        },
    }

df_new = spark.read.table(ds_config["bronze_table"])
changes = ds_config["changes"]
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]

logger.info("Silver layer CPI table configuration loaded")

['Subcouncil', 'B__30___1_', 'C_0_29', 'D_30_59', 'E_60_89', 'F_90_119', 'G_120_149', 'H_150_179', 'I_180_364', 'J_365_729', 'K_730_1094', 'L_1095_1459', 'M_1460_1824', 'N_1825_2189', 'O_2190_2554', 'P_2555_2919', 'Q_2920_3284', 'R___3285', 'Date']

df_new = spark.table(ds_config["bronze_table"])
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]

logger.info("\t- Silver layer Dam levels table configuration loaded")


# 3. Dataset Cleaning

## 3.1 Drop Columns

In [0]:
if changes["drop_columns"] and isinstance(col_config["columns_to_drop"], list):
    logger.info("Dropping column(s)")
    df_new = df_new.drop(*col_config["columns_to_drop"])
    logger.info("\t- Column(s) dropped")


## 3.2 Rename Headers

In [0]:
if changes["rename_headers"]:
    logger.info("Renaming headers")
    hdr_transformations = dict()

    if isinstance(hdr_config["column_mapping"], dict):

        df_new = df_new.select(
        [F.col(col).alias(hdr_config["column_mapping"].get(col, col)) for col in df_new.columns]
    )

    logger.info("\t- Header(s) renamed")

## 3.4 Trim Whitespace

In [0]:
if changes["trim"]:
    logger.info("Trimming column(s)")
    df_new = df_new.select([F.trim(F.col(col)).alias(col) for col in df_new.columns])
    logger.info("\t- Column(s) trimmed")

## 3.3 Column Cleanse

In [0]:
if changes["col_cleanse"]:
        logger.info("Cleaning column(s)")
        
        # eliminates overal result row
        df_new = df_new.filter(
                ~F.lower(F.col("subcouncil")).rlike("overall|Result")
        )

        # standardises date to - [hyphen] separator
        df_new = df_new.withColumn("date", F.regexp_replace(F.col("date"), "/",  "-"))

        # eliminates all empty data from 2024
        df_new = df_new.filter(
                ~F.col("date").rlike("2024")
        )
        logger.info("\t- Column(s) cleaned")

## 3.4 Unpivot Table

In [0]:
if changes["unpivot"]:
    logger.info("Unpivoting table")    
    exclude_cols = ["subcouncil", "date"]

    include_cols = [F.col(col) for col in df_new.columns if col not in exclude_cols]

    df_new = df_new.unpivot(
        ids = exclude_cols,
        values = include_cols,
        variableColumnName = "ageing_bucket_days",
        valueColumnName = "amount"
    )

    logger.info("\t- Table unpivoted")

## 3.5 Adding Primary key

In [0]:
if changes["add_id"]:
    logger.info("Adding ID(s)")   
    df_new = df_new.withColumn(
        "id",
        F.md5(
            F.concat_ws(
                "|",
                F.col("date"),
                F.col("subcouncil"),
                F.col("ageing_bucket_days"),
            )
        ),
    )
    logger.info("\t- ID(s) added")

## 3.6 Cast Data Types

In [0]:
if changes["cast_type"]:
    logger.info("Casting Datatype(s)")
    df_new = df_new.select([
            F.coalesce(
                F.try_to_date(F.col(col), "dd-MM-yyyy"),
                F.try_to_date(F.col(col), "yyyy-MM-dd")
            ).alias("date") if col == "date"
            else F.col(col).cast(col_config["data_types"][col]).alias(col) 
            for col in df_new.columns
           
     ])
    logger.info("\t- Datatype(s) casted")
    df_new.count()
    df_new.display()

## 3.7 Drop Duplicates

In [0]:
if changes["drop_duplicates"]:
      
    logger.info("Dropping Duplicate(s)")

    # Check all columns except 'date' for nulls
    excluded_cols = ["date", "id","ageing_bucket_days"]
    cols_to_check = [c for c in df_new.columns if c not in exclude_cols]

    # Count how many NULLs exist in each row across all columns
    null_count_expr = sum(
        [F.when(F.col(c).isNull(), 1).otherwise(0) for c in cols_to_check]
    )

    df_with_nulls = df_new.withColumn("null_count", null_count_expr)

    # Create a window grouped by date and dam_name, ordered by null_count ASCENDING least nulls wins.
    # (Lowest null count = rank 1)
    window_spec = Window.partitionBy("date", "subcouncil","ageing_bucket_days").orderBy(F.col("null_count").asc())

    # Filter to keep only the best row per date and clean up temporary columns
    df_deduped = (
        df_with_nulls.withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num", "null_count")
    )

    df_new = df_deduped
    logger.info("\t - Duplicate(s) Dropped")
df_new.count()
df_new.display()

# 4. Writing  To Silver Layer

In [0]:
# if changes["write_to_table"]:    
#     logger.info("Writing dam levels to Silver Table")
#     target_table_name = ds_config["silver_table"]

#     # 1. Register your cleaned DataFrame as a temporary view for SQL execution
#     df_new.createOrReplaceTempView("temp_source_data")

#     if spark.catalog.tableExists(target_table_name):
#         # 2. Run the native Databricks SQL Merge command
#         spark.sql(f"""
#             MERGE INTO {target_table_name} AS target
#             USING temp_source_data AS source
#             ON target.id = source.id
#             WHEN MATCHED THEN 
#                 UPDATE SET *
#             WHEN NOT MATCHED THEN 
#                 INSERT *
#         """)
#         logger.info("\t - Delta table successfully upserted.")
#     else:
#         # First-time run: Create the table
#         (
#             df_new.write
#             .format("delta")
#             .mode("overwrite")
#             .saveAsTable(target_table_name)
#         )
#         logger.info("\t - Target table didnot exist. Created new Delta table.")
#     logger.info("\t - Dam levels written to Silver Table")